In [2]:
pip install -U langchain-ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import ipywidgets as widgets
from IPython.display import display, Markdown
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage

import streamlit as st

# llm 모델 설정
llm = ChatOllama(
    model="qwen3.5:4b",
    base_url="http://127.0.0.1:11434",
    temperature=0.1,    # 창의성 및 무작위성
    reasoning=False,    # 추론 과정 생략
    num_ctx=4096,       # 모델이 기억할 수 있는 최대 토큰의 양
    num_predict=768     # 최대 출력 토큰 제한
)

# 임베딩 모델 설정
embeddings = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://127.0.0.1:11434",
)


In [15]:
# 대화 기록
system_message = SystemMessage(
    content="너는 친절한 AI 도우미야. 질문에 한국어로 명확하게 답변해."
)
messages = [system_message]

# 대화 표시 영역
chat_output = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        height="400px",
        overflow="auto",
        border="1px solid #ddd",
        padding="12px",
    )
)

# 입력창과 버튼
input_box = widgets.Textarea(
    placeholder="메시지를 입력한 뒤 전송을 누르세요.",
    layout=widgets.Layout(width="100%", height="80px"),
)
send_button = widgets.Button(
    description="전송",
    button_style="primary",
    icon="paper-plane",
)
clear_button = widgets.Button(description="대화 초기화")
status = widgets.Label(value="대화를 시작해보세요.")


# 메시지 화면 출력
def show_message(role, text):
    chat_output.append_display_data(
        Markdown(f"**{role}**\n\n{text}\n\n---")
    )


# 전송 버튼 처리
def send_message(_):
    text = input_box.value.strip()

    if not text:
        return

    send_button.disabled = True
    clear_button.disabled = True
    input_box.disabled = True
    status.value = "답변을 생성하고 있어요..."

    user_message = HumanMessage(content=text)
    show_message("나", text)

    try:
        # 이전 대화와 현재 질문 전달
        response = llm.invoke(messages + [user_message])

        show_message("AI", response.content)

        # 다음 질문에서도 참고하도록 기록
        messages.extend([user_message, response])
        input_box.value = ""
        status.value = "답변 완료"

    except Exception as e:
        chat_output.append_stdout(f"오류: {e}\n")
        status.value = "연결 상태를 확인한 뒤 다시 전송해주세요."

    finally:
        send_button.disabled = False
        clear_button.disabled = False
        input_box.disabled = False


# 초기화 버튼 처리
def clear_chat(_):
    messages[:] = [system_message]
    chat_output.clear_output()
    input_box.value = ""
    status.value = "대화를 초기화했어요."


send_button.on_click(send_message)
clear_button.on_click(clear_chat)

# 주피터 셀 안에 UI 표시
display(
    widgets.VBox([
        widgets.HTML("<h3>💬 Ollama 챗봇</h3>"),
        chat_output,
        input_box,
        widgets.HBox([send_button, clear_button]),
        status,
    ])
)

In [16]:
%%writefile app.py
import streamlit as st
from langchain_ollama import ChatOllama

st.set_page_config(page_title="Ollama 챗봇", page_icon="💬")
st.title("💬 Ollama 챗봇")


@st.cache_resource
def get_llm():
    return ChatOllama(
        model="qwen3.5:4b",
        base_url="http://127.0.0.1:11434",
        temperature=0.1,
        reasoning=False,
        num_ctx=4096,
        num_predict=768,
    )


llm = get_llm()

if "messages" not in st.session_state:
    st.session_state.messages = []

with st.sidebar:
    st.write("모델: qwen3.5:4b")

    if st.button("대화 초기화"):
        st.session_state.messages = []
        st.rerun()

# 이전 대화 표시
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# 사용자 입력
if prompt := st.chat_input("메시지를 입력하세요."):
    user_message = {"role": "user", "content": prompt}

    with st.chat_message("user"):
        st.markdown(prompt)

    request_messages = [
        {
            "role": "system",
            "content": "너는 친절한 AI 도우미야. 한국어로 명확하게 답변해.",
        },
        *st.session_state.messages,
        user_message,
    ]

    with st.chat_message("assistant"):
        try:
            with st.spinner("답변 생성 중..."):
                response = llm.invoke(request_messages)

            st.markdown(response.content)

            st.session_state.messages.extend([
                user_message,
                {"role": "assistant", "content": response.content},
            ])

        except Exception as e:
            st.error(f"답변 생성 실패: {e}")

Writing app.py


In [17]:
import subprocess
import sys

# 이 셀을 다시 실행하면 기존 프로세스 종료
if "streamlit_process" in globals():
    streamlit_process.terminate()
    streamlit_process.wait()

with open("streamlit.log", "w") as log:
    streamlit_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", "app.py",
            "--server.address=0.0.0.0",
            "--server.port=8501",
            "--server.headless=true",
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
    )

print("실행 요청 완료. 잠시 후 다음 셀을 실행하세요.")

실행 요청 완료. 잠시 후 다음 셀을 실행하세요.


In [19]:
from IPython.display import display, IFrame

display(IFrame(
    src="http://localhost:8501",
    width="100%",
    height=750,
))